In [ ]:
import pandas as pd
import io

raw_data = """
HOUR,NB_LOAD,NB_DEMAND,ISO_NE,NMISA,QUEBEC,NOVA_SCOTIA,PEI
2026-03-01 00:00,1659,1703,238,71,-301,-30,47
2026-03-01 01:00,1616,1654,222,69,-401,-30,67
2026-03-01 02:00,1599,1637,199,67,-402,-30,83
2026-03-01 03:00,1605,1642,163,68,-446,-30,95
2026-03-01 04:00,1627,1665,155,70,-467,-30,111
2026-03-01 05:00,1661,1702,125,76,-521,-30,123
2026-03-01 06:00,1710,1753,116,80,-532,-30,130
2026-03-01 07:00,1760,1805,116,82,-551,-30,152
2026-03-01 08:00,1849,1899,117,85,-601,-30,144
2026-03-01 09:00,1951,2005,127,83,-674,-30,150
2026-03-01 10:00,1978,2034,126,79,-652,-30,148
2026-03-01 11:00,1985,2041,106,67,-640,-30,160
2026-03-01 12:00,2006,2063,101,60,-701,-30,161
2026-03-01 13:00,2011,2068,101,56,-726,-30,119
2026-03-01 14:00,2017,2072,111,54,-677,-30,137
2026-03-01 15:00,2003,2060,119,39,-667,-30,141
2026-03-01 16:00,2000,2057,126,40,-690,-30,121
2026-03-01 17:00,2053,2116,47,43,-740,-30,140
2026-03-01 18:00,2111,2185,106,74,-890,-29,155
2026-03-01 19:00,2212,2293,138,87,-950,0,175
2026-03-01 20:00,2254,2336,163,85,-901,0,168
2026-03-01 21:00,2271,2355,259,84,-990,-29,155
2026-03-01 22:00,2254,2337,274,72,-991,-30,131
2026-03-01 23:00,2213,2290,289,71,-891,-30,97
"""

df = pd.read_csv(io.StringIO(raw_data))
df.head()


,HOUR,NB_LOAD,NB_DEMAND,ISO_NE,NMISA,QUEBEC,NOVA_SCOTIA,PEI
0,2026-03-01 00:00,1659,1703,238,71,-301,-30,47
1,2026-03-01 01:00,1616,1654,222,69,-401,-30,67
2,2026-03-01 02:00,1599,1637,199,67,-402,-30,83
3,2026-03-01 03:00,1605,1642,163,68,-446,-30,95
4,2026-03-01 04:00,1627,1665,155,70,-467,-30,111


In [ ]:
import numpy as np
import pandas as pd

x = df["NB_LOAD"].values.astype(float)

v = np.diff(x, prepend=x[0])
a = np.diff(v, prepend=v[0])

eps = 1e-12

# Curvature
kappa = np.abs(v * a) / (np.abs(v)**3 + eps)

# ACCR
ACCR = kappa / (np.abs(a) + eps)

# Updated baseline_kappa (no deprecated fillna)
baseline_kappa = (
    pd.Series(kappa)
    .rolling(5, center=True)
    .mean()
    .bfill()
    .ffill()
)

# Funnel 1
C1 = kappa - baseline_kappa
R = (np.abs(v)**2) / (np.abs(a) + eps)
F1 = C1 * R

# Updated P (no deprecated fillna)
P = (
    pd.Series(v)
    .rolling(7, center=True)
    .std()
    .bfill()
    .ffill()
)

C2 = np.abs(np.diff(np.sign(v), prepend=np.sign(v[0])))
F2 = P * C2

# Collapse boundary
t_c = np.argmax(ACCR)

# Normalize funnels
sF1 = (F1 - F1.min()) / (F1.max() - F1.min() + eps)
sF2 = (F2 - F2.min()) / (F2.max() - F2.min() + eps)

tau = 0.15

# Early warning trigger
t1_candidates = np.where((sF1 > tau) & (np.arange(len(F1)) < t_c))[0]
t1 = t1_candidates[0] if len(t1_candidates) > 0 else None

# Reactive trigger
t2_candidates = np.where((sF2 > tau) & (np.arange(len(F2)) > t_c))[0]
t2 = t2_candidates[0] if len(t2_candidates) > 0 else None

# Updated severity (no deprecated trapz)
severity = np.trapezoid(ACCR[t_c:], dx=1.0)

print("t1 =", t1)
print("t_c =", t_c)
print("t2 =", t2)
print("Predictive Horizon H1 =", t_c - t1)
print("Reactive Horizon H2 =", t2 - t_c)
print("Severity =", severity)


t1 = 0
t_c = 16
t2 = 17
Predictive Horizon H1 = 16
Reactive Horizon H2 = 1
Severity = 0.0640915995447777
